# Bewerbungsanalyse

Durchsucht den Ordner `Anschreiben_Alt` (Unterordner `Alt` und `Neu`) nach Bewerbungsschreiben (`Anschreiben_*.pdf`, `Motivationsschreiben_*.pdf`) und wertet aus:

- Gesamtanzahl der Bewerbungen
- Anzahl der Bewerbungen pro Monat
- Firmen, an die pro Monat beworben wurde
- Genaues Absendedatum je Bewerbung

Datum und Firma werden primär aus dem Dateinamen geparst (z.B. `Anschreiben_Krones_16.06.26.pdf`). Enthält ein Dateiname kein Datum, wird ersatzweise das Änderungsdatum der Datei verwendet.

In [1]:
import os
import re
from datetime import datetime

import pandas as pd

BASE_DIR = r"C:\Users\funke\OneDrive\Bewerbungen\Bewerbungen\Anschreiben_Alt"
SUBFOLDERS = ["Alt", "Neu"]

MONTH_NAMES_DE = {
    1: "Januar", 2: "Februar", 3: "März", 4: "April", 5: "Mai", 6: "Juni",
    7: "Juli", 8: "August", 9: "September", 10: "Oktober", 11: "November", 12: "Dezember",
}

print(f"Basisordner: {BASE_DIR}")
print(f"Unterordner: {SUBFOLDERS}")

Basisordner: C:\Users\funke\OneDrive\Bewerbungen\Bewerbungen\Anschreiben_Alt
Unterordner: ['Alt', 'Neu']


## Dateien einlesen und parsen

Firmenname und Datum werden aus dem Dateinamen extrahiert (Muster `Anschreiben_<Firma>_<TT.MM.JJ(JJ)>.pdf` bzw. `Motivationsschreiben_<Firma>_<TT.MM.JJ(JJ)>.pdf`). Fehlt das Datum im Dateinamen, wird das Änderungsdatum der Datei als Fallback verwendet (in der Ausgabe als "(Dateidatum)" markiert).

In [2]:
FILENAME_DATE_RE = re.compile(r"(\d{1,2})\.(\d{1,2})\.(\d{2,4})")
FILENAME_DATE_TYPO_RE = re.compile(r"(\d{1,2})\.(\d{2})(\d{2})(?!\d)")
PREFIX_RE = re.compile(r"^(Anschreiben|Motivationsschreiben)[\s_]*", re.IGNORECASE)


def normalize_year(year: str) -> int:
    year = "20" + year if len(year) == 2 else year
    return int(year)


def parse_date(day, month, year):
    try:
        return datetime(normalize_year(year), int(month), int(day))
    except ValueError:
        return None


def extract_date_from_filename(stem: str):
    for pattern in (FILENAME_DATE_RE, FILENAME_DATE_TYPO_RE):
        m = pattern.search(stem)
        if m:
            d = parse_date(*m.groups())
            if d:
                return d
    return None


def parse_filename(stem: str):
    prefix_match = PREFIX_RE.match(stem)
    rest = stem[prefix_match.end():] if prefix_match else stem

    date_match = FILENAME_DATE_RE.search(rest) or FILENAME_DATE_TYPO_RE.search(rest)
    firma_part = rest[: date_match.start()] if date_match else rest

    firma = firma_part.replace("_", " ").replace("�", "")
    firma = re.sub(r"\s+", " ", firma).strip(" -")
    return firma or "(unbekannt)"


def collect_bewerbungen(base_dir, subfolders):
    rows = []
    for folder in subfolders:
        folder_path = os.path.join(base_dir, folder)
        if not os.path.isdir(folder_path):
            continue
        for filename in sorted(os.listdir(folder_path)):
            if not filename.lower().endswith(".pdf"):
                continue
            stem = os.path.splitext(filename)[0]
            if not PREFIX_RE.match(stem):
                continue  # z.B. Kombizertifikat o.ä. ueberspringen

            full_path = os.path.join(folder_path, filename)
            firma = parse_filename(stem)

            date_obj = extract_date_from_filename(stem)
            datum_quelle = "Dateiname"
            if date_obj is None:
                date_obj = datetime.fromtimestamp(os.path.getmtime(full_path))
                datum_quelle = "Dateidatum"

            rows.append({
                "ordner": folder,
                "firma": firma,
                "datum": date_obj,
                "datum_quelle": datum_quelle,
                "dateiname": filename,
            })
    return rows


rows = collect_bewerbungen(BASE_DIR, SUBFOLDERS)
df = pd.DataFrame(rows).sort_values("datum").reset_index(drop=True)
df["jahr"] = df["datum"].dt.year
df["monat_num"] = df["datum"].dt.month
df["monat"] = df.apply(lambda r: f"{r['jahr']}-{r['monat_num']:02d} ({MONTH_NAMES_DE[r['monat_num']]})", axis=1)
df["datum_str"] = df["datum"].dt.strftime("%d.%m.%Y")

print(f"{len(df)} Bewerbungsdateien gefunden.")
df.head()

133 Bewerbungsdateien gefunden.


,ordner,firma,datum,datum_quelle,dateiname,jahr,monat_num,monat,datum_str
0,Alt,TECVIA,2025-09-29 11:19:02,Dateidatum,Motivationsschreiben_TECVIA.pdf,2025,9,2025-09 (September),29.09.2025
1,Alt,Attek,2025-09-29 12:30:30,Dateidatum,Motivationsschreiben_Attek.pdf,2025,9,2025-09 (September),29.09.2025
2,Alt,Intend,2025-09-29 15:32:04,Dateidatum,Motivationsschreiben_Intend.pdf,2025,9,2025-09 (September),29.09.2025
3,Alt,FIOSYSTEMS,2025-09-29 15:54:52,Dateidatum,Motivationsschreiben_FIOSYSTEMS.pdf,2025,9,2025-09 (September),29.09.2025
4,Alt,ADVERGYGmbH,2025-10-06 11:48:52,Dateidatum,Motivationsschreiben_ADVERGYGmbH.pdf,2025,10,2025-10 (Oktober),06.10.2025


## Gesamtanzahl der Bewerbungen

In [3]:
gesamt = len(df)
zeitraum_von = df["datum"].min().strftime("%d.%m.%Y")
zeitraum_bis = df["datum"].max().strftime("%d.%m.%Y")

print(f"Gesamtanzahl Bewerbungen: {gesamt}")
print(f"Zeitraum: {zeitraum_von} bis {zeitraum_bis}")

anzahl_ohne_datum = (df["datum_quelle"] == "Dateidatum").sum()
if anzahl_ohne_datum:
    print(f"Hinweis: bei {anzahl_ohne_datum} Datei(en) kein Datum im Dateinamen gefunden – Änderungsdatum der Datei wurde verwendet.")

Gesamtanzahl Bewerbungen: 133
Zeitraum: 29.09.2025 bis 30.08.2026
Hinweis: bei 40 Datei(en) kein Datum im Dateinamen gefunden – Änderungsdatum der Datei wurde verwendet.


## Anzahl der Bewerbungen pro Monat

In [4]:
pro_monat = df.groupby("monat").size().reset_index(name="anzahl").sort_values("monat")
pro_monat

,monat,anzahl
0,2025-09 (September),4
1,2025-10 (Oktober),17
2,2025-11 (November),4
3,2025-12 (Dezember),5
4,2026-01 (Januar),5
5,2026-02 (Februar),6
6,2026-03 (März),7
7,2026-04 (April),9
8,2026-05 (Mai),31
9,2026-06 (Juni),14


## Firmen pro Monat

In [5]:
for monat, gruppe in df.sort_values(["monat", "datum"]).groupby("monat"):
    firmen = ", ".join(f"{r.firma} ({r.datum_str})" for r in gruppe.itertuples())
    print(f"{monat} – {len(gruppe)} Bewerbung(en):")
    print(f"  {firmen}\n")

2025-09 (September) – 4 Bewerbung(en):
  TECVIA (29.09.2025), Attek (29.09.2025), Intend (29.09.2025), FIOSYSTEMS (29.09.2025)

2025-10 (Oktober) – 17 Bewerbung(en):
  ADVERGYGmbH (06.10.2025), GEFASOFT (06.10.2025), evopro (06.10.2025), SIITechnologiesGmbH (07.10.2025), Alfatraining (09.10.2025), Krones (13.10.2025), CodeCompass (20.10.2025), GFN (20.10.2025), InfoTip (21.10.2025), IQVIA (21.10.2025), Kaupa (23.10.2025), FinanzIT (24.10.2025), Hallowelt (24.10.2025), ROPA (28.10.2025), EA (29.10.2025), Lersch (29.10.2025), niwadev (30.10.2025)

2025-11 (November) – 4 Bewerbung(en):
  HAPEKO (14.11.2025), RHAPSODY (19.11.2025), INFOMOTION (19.11.2025), WattFox (27.11.2025)

2025-12 (Dezember) – 5 Bewerbung(en):
  PostHog (02.12.2025), LinoPro (08.12.2025), KA Resources (24.12.2025), VESTERLING (24.12.2025), IVUSoftwareentwicklungGmbH (24.12.2025)

2026-01 (Januar) – 5 Bewerbung(en):
  Piening (05.01.2026), Brunel (08.01.2026), HiQ (09.01.2026), GUS (28.01.2026), Nexis (28.01.2026)

202

## Alle Bewerbungen im Detail (chronologisch)

In [6]:
detail = df[["datum_str", "firma", "ordner", "datum_quelle", "dateiname"]].rename(columns={
    "datum_str": "Datum",
    "firma": "Firma",
    "ordner": "Ordner",
    "datum_quelle": "Datumsquelle",
    "dateiname": "Dateiname",
})
pd.set_option("display.max_rows", None)
detail

,Datum,Firma,Ordner,Datumsquelle,Dateiname
0,29.09.2025,TECVIA,Alt,Dateidatum,Motivationsschreiben_TECVIA.pdf
1,29.09.2025,Attek,Alt,Dateidatum,Motivationsschreiben_Attek.pdf
2,29.09.2025,Intend,Alt,Dateidatum,Motivationsschreiben_Intend.pdf
3,29.09.2025,FIOSYSTEMS,Alt,Dateidatum,Motivationsschreiben_FIOSYSTEMS.pdf
4,06.10.2025,ADVERGYGmbH,Alt,Dateidatum,Motivationsschreiben_ADVERGYGmbH.pdf
5,06.10.2025,GEFASOFT,Alt,Dateidatum,Motivationsschreiben_GEFASOFT.pdf
6,06.10.2025,evopro,Alt,Dateidatum,Motivationsschreiben_evopro.pdf
7,07.10.2025,SIITechnologiesGmbH,Alt,Dateidatum,Motivationsschreiben_SIITechnologiesGmbH.pdf
8,09.10.2025,Alfatraining,Alt,Dateidatum,Motivationsschreiben_Alfatraining.pdf
9,13.10.2025,Krones,Alt,Dateidatum,Motivationsschreiben_Krones.pdf
